# Fase 6 & 7: Explicabilidad con SHAP y Auditoría de Equidad (Fairness)

## Contenido del Notebook
1. **Evaluación Final en el Conjunto de Prueba Aislado (Test Set)**.
2. **Explicabilidad Global y Local con SHAP** (`shap.TreeExplainer`).
3. **Desglose de Importancia: Factores Clínicos vs. Estilo de Vida**.
4. **Auditoría de Subgrupos y Equidad (*Fairness Audit*)**:
   - Desempeño por Sexo biológico (`Male` vs. `Female`).
   - Desempeño por Estrato Socioeconómico (`income_level`: Low, Middle, High).
   - Desempeño por Rango de Edad (<45, 45-60, >60 años).


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.config import TARGET_COL, FIGURES_DIR
from src.data_loader import load_processed_splits
from src.models import build_lightgbm_pipeline
from src.evaluation import compute_metrics, audit_subgroup_fairness
from src.explainability import compute_tree_shap_values, aggregate_group_importance
from src.fairness import evaluate_equal_opportunity

train_df, val_df, test_df = load_processed_splits()
X_train, y_train = train_df.drop(columns=[TARGET_COL]), train_df[TARGET_COL]
X_test, y_test = test_df.drop(columns=[TARGET_COL]), test_df[TARGET_COL]

# Ajustar modelo en Train+Val (85k) o directamente en Train para evaluación final
print("Entrenando modelo final LightGBM...")
final_model = build_lightgbm_pipeline(n_estimators=300, learning_rate=0.05)
final_model.fit(X_train, y_train)

test_probas = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probas >= 0.50).astype(int)

test_metrics = compute_metrics(y_test, test_preds, test_probas)
print("=== Métricas en Test (15,000 registros independientes) ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")
    else:
        print(f"{k:25s}: {v}")


## 1. Explicabilidad con SHAP (*TreeExplainer*)
Calculamos los valores SHAP sobre una muestra representativa del conjunto de prueba (ej. 2,000 registros para alta velocidad y precisión numérica).


In [ ]:
sample_test = X_test.sample(2000, random_state=42)
shap_values = compute_tree_shap_values(final_model, sample_test)

# Beeswarm Summary Plot
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("Impacto Global de Características (SHAP Beeswarm)", fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_beeswarm.png", dpi=300, bbox_inches='tight')
plt.show()


---
## 2. Ejercicios Guiados para el Alumno (TODOs)


### Pregunta Central de Investigación
> *¿Cuál es el factor de riesgo de sufrir una enfermedad cardíaca, evaluando el impacto diferenciado entre factores clínicos y factores de estilo de vida?*

1. Utiliza la función `aggregate_group_importance(shap_values)` de `src.explainability`.
2. Genera un gráfico de pastel (*pie chart*) o de barras mostrando el **porcentaje de contribución relativa**:
   - Factores Clínicos (Marcadores Clínicos + Historial Médico)
   - Estilo de Vida (Dieta, Ejercicio, Tabaquismo, Alcohol, Sueño, Estrés)
   - Dispositivos Vestibles (HRV, Frecuencia Cardíaca en Reposo)
   - Demografía (Edad, Sexo, Región, Ingresos)


In [ ]:
# TODO: Ejecuta aggregate_group_importance y grafica el porcentaje de impacto
group_summary = aggregate_group_importance(shap_values)
print("=== Resumen Agrupado de Importancia SHAP ===")
display(group_summary)

plt.figure(figsize=(8, 6))
plt.pie(
    group_summary['Percentage_Contribution'],
    labels=group_summary['Group'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#2a78d6', '#eb6834', '#1baf7a', '#eda100'],
    explode=[0.05, 0.05, 0.05, 0.05]
)
plt.title('Contribución Relativa de Factores en el Riesgo Cardíaco', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_group_importance.png", dpi=300, bbox_inches='tight')
plt.show()


### Auditoría de Equidad y Subgrupos (*Fairness Audit*)
Evalúa si el modelo presenta sesgos o disparidad en el diagnóstico médico según:
1. **Sexo biológico**: `test_df['sex']` (Male vs Female).
2. **Nivel de Ingresos**: `test_df['income_level']` (Low vs Middle vs High).
3. **Grupo de Edad**: Crea rangos etarios (<45, 45-60, >60 años) a partir de `test_df['age']`.

**Preguntas clave para el reporte:**
- ¿Existe una brecha significativa en el *Recall* (Sensibilidad) entre hombres y mujeres?
- ¿El modelo subestima o sobreestima el riesgo en pacientes de bajos ingresos (*Low income*)?
- Utiliza `audit_subgroup_fairness` o `evaluate_equal_opportunity` para cuantificar estas métricas.


In [ ]:
# TODO: Realiza la auditoría de subgrupos por Sexo, Ingresos y Edad
print("=== 1. Auditoría de Equidad por Sexo Biológico ===")
audit_sex = audit_subgroup_fairness(y_test, test_probas, test_df['sex'])
display(audit_sex)

print("\n=== 2. Auditoría de Equidad por Nivel de Ingresos ===")
audit_income = audit_subgroup_fairness(y_test, test_probas, test_df['income_level'])
display(audit_income)

print("\n=== 3. Auditoría de Equidad por Grupos de Edad ===")
age_bins = pd.cut(test_df['age'], bins=[0, 45, 60, 100], labels=['<45 años', '45-60 años', '>60 años'])
audit_age = audit_subgroup_fairness(y_test, test_probas, age_bins)
display(audit_age)
